# 04 — Phân tích khám phá & trực quan hóa khách hàng rời bỏ Cell2Cell

**Vai trò:** chuyên viên phân tích dữ liệu của công ty viễn thông Cell2Cell.

**Bài toán kinh doanh:** trong mẫu dữ liệu, gần 3 trong 10 khách rời bỏ dịch vụ. Ban giám đốc cần biết:

1. Tình trạng rời bỏ phổ biến đến đâu và gây thiệt hại bao nhiêu?
2. Tín hiệu nào (vòng đời, thiết bị, hành vi dịch vụ, địa lý, nhân khẩu) đi cùng rời bỏ?
3. Ai đang có nguy cơ rời bỏ và nên ưu tiên giữ ai?
4. Nên thử chính sách giữ chân nào?

**Cách chạy:** chạy `python work/viz/prepare_viz_data.py` trước (tạo `work/build/viz_customers.parquet`), sau đó Khởi động lại nhân tính toán → Chạy tất cả các ô.
Mọi biểu đồ được vẽ bằng hàm trong `work/viz/charts.py`, cùng thư viện với bảng điều khiển, nên số liệu trên sổ phân tích và bảng điều khiển luôn khớp.
Tiêu đề mỗi biểu đồ là một **câu kết luận** sinh tự động từ số liệu.

## Thiết lập

In [ ]:
from pathlib import Path
from base64 import b64encode
import json
import sys
import importlib

from IPython.display import Markdown, display

import pandas as pd

# Tìm thư mục gốc dự án từ thư mục làm việc (chạy được từ root hoặc từ work/viz).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "cell2celltrain.csv").exists())
sys.path.insert(0, str(ROOT / "work" / "viz"))
import charts as c
c = importlib.reload(c)
import prepare_viz_data as preparation
preparation = importlib.reload(preparation)

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 20)
dashboard_figures = {}

def show(key):
    """Vẽ biểu đồ theo mã trong charts.FIGURES và lưu PNG vào reports/figures."""
    fig = c.FIGURES[key](df)
    dashboard_figures[key] = fig
    image_path = FIG_DIR / f"{key}.png"
    fig.write_image(image_path, width=1000, scale=1.5)
    # Gửi trực tiếp hai định dạng cho Jupyter/VS Code; không phụ thuộc bộ
    # kiểm tra nbformat của fig.show(). PNG vẫn dùng để xem trên GitHub.
    display({
        "application/vnd.plotly.v1+json": json.loads(fig.to_json()),
        "image/png": b64encode(image_path.read_bytes()).decode("ascii"),
    }, raw=True)

def explain(key):
    figure = dashboard_figures[key]
    if not figure.data:
        display(Markdown(figure.layout.annotations[0].text))
        return
    title = figure.layout.title.text.split("<br>")[0].replace("<b>", "").replace("</b>", "")
    display(Markdown("**Nhận xét:** " + title + "."))


## Dữ liệu và bộ biến thống nhất

Mô hình và toàn bộ EDA chỉ sử dụng cùng **10 biến đầu vào**. `Churn` là nhãn cần dự đoán; mã khách chỉ phục vụ đối soát. Các nhóm giá trị và điểm rủi ro được suy ra từ bộ 10, không phải predictor bổ sung. CSV nguồn giữ nguyên cấu trúc.

| Biến mô hình | Ý nghĩa | Biểu đồ trực tiếp |
|---|---|---|
| `MonthsInService` | Thời gian sử dụng | C05, C05b, C14 |
| `CurrentEquipmentDays` | Tuổi thiết bị | C03, C04, C06, C07 |
| `HandsetWebCapable` | Hỗ trợ truy cập web | C09 |
| `MonthlyRevenue` | Doanh thu tháng | C04, C04b |
| `TotalRecurringCharge` | Phí định kỳ | C04c, C04d |
| `CreditRating` | Hạng tín dụng | C16 |
| `MonthlyMinutes` | Phút gọi hàng tháng | C12 |
| `PercChangeMinutes` | Mức thay đổi phút gọi | C10 |
| `PercChangeRevenues` | Mức thay đổi doanh thu | C11 |
| `PeakCallsInOut` | Cuộc gọi giờ cao điểm | C13 |

In [ ]:
raw = pd.read_csv(ROOT / "data" / "cell2celltrain.csv", na_values=["NA"])
source_column_count = raw.shape[1]
raw = raw[["CustomerID", "Churn", *preparation.EDA_COLUMNS]].copy()
print(f"Nguồn: {c.fmt_n(len(raw))} khách × {source_column_count} cột; mô hình chọn {len(c.C18_FEATURES)} predictor.")
print(f"EDA khảo sát {len(preparation.EDA_COLUMNS)} trường; mã khách trùng: {raw.CustomerID.duplicated().sum()}")
data_types = raw[preparation.EDA_COLUMNS].dtypes.astype(str).map({"int64": "Số nguyên", "float64": "Số thực", "object": "Văn bản"})
display(data_types.value_counts().rename_axis("Kiểu dữ liệu").to_frame("Số trường EDA"))
c.display_table(raw.head())

In [ ]:
c.display_table(raw[preparation.EDA_COLUMNS].describe().T[["count", "mean", "50%", "min", "max"]].round(1))

## Làm sạch và chuẩn bị dữ liệu

Chỉ chọn 10 biến mô hình. Nhận diện giá trị thiếu; loại số âm bất thường ở các đại lượng không thể âm; giữ số âm ở hai mức thay đổi. Các band thời gian, tuổi máy, phút gọi, phí định kỳ và giờ cao điểm đều tạo từ bộ 10.

Hai mức thay đổi được phân nhóm theo dấu: Giảm, Không đổi, Tăng; không diễn giải chúng là phần trăm khi chưa xác nhận đơn vị. Không dùng cột ngoài mô hình để lọc mẫu, chia phân khúc hoặc vẽ biểu đồ.

In [ ]:
import importlib
import prepare_viz_data as preparation
preparation = importlib.reload(preparation)
cleaned_raw, negative_counts = preparation.remove_invalid_negative_rows(raw)
removed_count = len(raw) - len(cleaned_raw)
metrics_path = ROOT / "work" / "build" / "model_metrics.json"
model_info = json.loads(metrics_path.read_text(encoding="utf-8")) if metrics_path.exists() else {}
if preparation.MART.exists() and model_info.get("features") != c.C18_FEATURES:
    raise ValueError("Điểm mô hình chưa dùng đúng bộ 10 biến. Hãy chạy notebook 03 trước.")
df = preparation.prepare_data(cleaned_raw)
df.to_parquet(ROOT / "work" / "build" / "viz_customers.parquet", index=False)
k = c.kpis(df)
print(f"Đã loại {c.fmt_n(removed_count)} dòng có số âm bất thường từ {c.fmt_n(len(raw))} khách nguồn.")
print(f"Bảng EDA: {c.fmt_n(len(df))} khách · {c.fmt_n(k['churners'])} rời bỏ · {c.fmt_pct(k['rate'], 2)}")
print(f"Mô hình: {len(model_info.get('features', []))} biến · {model_info.get('model_version', 'chưa có điểm')}")
c.display_table(df[["TenureBand", "EquipmentAgeBand", "ChargeBand", "MinutesBand", "UsageTrendBand", "RevenueTrendBand", "PeakCallsBand", "ActionSegment"]].describe())

### C02 — Dữ liệu thiếu ở đâu?

In [ ]:
show("C02_du_lieu_thieu")

In [ ]:
explain('C02_du_lieu_thieu')

## 2.1. Tổng quan dữ liệu

### C01 — Tình trạng rời bỏ phổ biến đến đâu?

In [ ]:
show("C01_ty_le_churn")

In [ ]:
explain('C01_ty_le_churn')

## 2.2. Phân tích thời gian sử dụng và thiết bị

### C03 — Phân bố tuổi thiết bị khác nhau thế nào giữa hai nhóm?

In [ ]:
show("C03_phan_bo_tuoi_may")

In [ ]:
explain('C03_phan_bo_tuoi_may')

### C05 — Tỷ lệ rời bỏ thay đổi thế nào theo thời gian sử dụng?

In [ ]:
show("C05_vong_doi")
show("C05b_so_khach")

In [ ]:
explain('C05_vong_doi')

### C14 — Khách rời bỏ tập trung ở giai đoạn sử dụng nào?

Giữ treemap với nhóm thời gian sử dụng. Diện tích là số khách rời bỏ, màu là tỷ lệ rời bỏ; không dùng chiều khu vực.

In [ ]:
show("C14_treemap_thoi_gian")

In [ ]:
explain('C14_treemap_thoi_gian')

### C06 — Tuổi thiết bị ảnh hưởng ra sao?

In [ ]:
show("C06_tuoi_thiet_bi")

In [ ]:
explain('C06_tuoi_thiet_bi')

### C07 — Thời gian sử dụng và tuổi máy kết hợp thế nào?

In [ ]:
show("C07_heatmap_tenure_thiet_bi")

In [ ]:
explain('C07_heatmap_tenure_thiet_bi')

### C09 — Thiết bị hỗ trợ web có tỷ lệ rời bỏ khác nhau không?

Giữ grouped bar Có/Không cho biến hỗ trợ web.

In [ ]:
show("C09_dac_diem_thiet_bi")

In [ ]:
explain('C09_dac_diem_thiet_bi')

## 2.3. Phân tích doanh thu, cước phí và mức sử dụng

### C04 — Giá trị ngoại lệ và độ phân tán ở các biến chính?

In [ ]:
show("C04_box_plot")

In [ ]:
explain('C04_box_plot')

### C04b — Phân bố doanh thu tháng khác nhau thế nào giữa khách rời bỏ và ở lại?

Histogram chồng chuẩn hóa theo từng nhóm; chỉ ẩn phần đuôi trên khi vẽ để dễ quan sát.

In [ ]:
show("C04b_phan_bo_doanh_thu")

In [ ]:
explain("C04b_phan_bo_doanh_thu")

### C04c — Phân bố phí định kỳ của hai nhóm có khác nhau không?

Giữ cùng form histogram chồng, màu rời bỏ và ở lại thống nhất với các biểu đồ trước.

In [ ]:
show("C04c_phan_bo_phi_dinh_ky")

In [ ]:
explain("C04c_phan_bo_phi_dinh_ky")

### C04d — Tỷ lệ rời bỏ khác nhau thế nào theo mức phí định kỳ?

Bar có baseline toàn mẫu; các nhóm phí ≤30, trên 30 đến 60 và trên 60 theo đơn vị nguồn. Giá trị phí thiếu không tham gia so sánh nhóm.

In [ ]:
show("C04d_churn_phi_dinh_ky")

In [ ]:
explain("C04d_churn_phi_dinh_ky")

### C12 — Tỷ lệ rời bỏ theo mức sử dụng phút gọi

Bubble chỉ dùng nhóm `MonthlyMinutes`, tỷ lệ rời bỏ và số khách; không còn chiều phân màu theo phút vượt gói.

In [ ]:
show("C12_muc_su_dung")

In [ ]:
explain('C12_muc_su_dung')

## 2.4. Phân tích biến động mức sử dụng và doanh thu

### C10 — Tỷ lệ rời bỏ khi phút gọi giảm, không đổi hoặc tăng

Bar theo dấu của `PercChangeMinutes`, kèm baseline toàn mẫu. Giá trị thiếu không tham gia so sánh nhóm.

In [ ]:
show("C10_bien_dong_phut_goi")

In [ ]:
explain('C10_bien_dong_phut_goi')

### C11 — Tỷ lệ rời bỏ khi doanh thu giảm, không đổi hoặc tăng

Bar theo dấu của `PercChangeRevenues`, kèm baseline toàn mẫu. Chỉ mô tả mức thay đổi theo dữ liệu nguồn, không giả định đơn vị phần trăm.

In [ ]:
show("C11_bien_dong_doanh_thu")

In [ ]:
explain('C11_bien_dong_doanh_thu')

## 2.5. Phân tích cuộc gọi giờ cao điểm

### C13 — Tỷ lệ rời bỏ theo số cuộc gọi giờ cao điểm

Bar ngang theo nhóm của `PeakCallsInOut`: ≤50, trên 50 đến 150, trên 150 đến 300 và trên 300. Đường đứt là churn trung bình của toàn mẫu.

In [ ]:
show("C13_goi_gio_cao_diem")

In [ ]:
explain('C13_goi_gio_cao_diem')

## 2.6. Phân tích hạng tín dụng khách hàng

### C16 — Hạng tín dụng liên quan thế nào đến rời bỏ?

In [ ]:
show("C16_tin_dung")

In [ ]:
explain('C16_tin_dung')

## 2.7. Phân tích tương quan giữa các biến

### C18 — 10 biến chính tương quan với rời bỏ và với nhau như thế nào?

Chỉ xét **10 biến đầu vào của mô hình hiện tại**: thời gian sử dụng, tuổi thiết bị, hỗ trợ web, doanh thu tháng, phí định kỳ, hạng tín dụng, phút gọi mỗi tháng, thay đổi phút gọi, thay đổi doanh thu và cuộc gọi giờ cao điểm. Thêm nhãn **Rời bỏ** để đối chiếu, tạo ma trận 11 × 11.

**Cách tính:** tương quan Pearson; hỗ trợ web Có = 1, Không = 0; hạng tín dụng từ 1 (cao nhất) đến 7 (thấp nhất). Giá trị thiếu/không rõ được bỏ theo từng cặp biến. Mã hóa chỉ phục vụ biểu đồ này; mô hình hiện tại cũng chỉ dùng 10 biến này.

In [ ]:
import importlib
importlib.reload(c)
show("C18_tuong_quan")

In [ ]:
c18_corr = c.correlation_matrix(df)
c18_target = c18_corr[c.TARGET].drop(c.TARGET).dropna()
if not c18_target.empty:
    c18_top = c18_target.abs().idxmax()
    c18_r = f"{c18_target[c18_top]:.2f}".replace(".", ",")
    display(Markdown(
        f"**Nhận xét:** trong **{len(c.C18_FEATURES)} biến chính**, **{c.field_label(c18_top)}** có tương quan tuyến tính "
        f"lớn nhất về độ lớn với rời bỏ (**r = {c18_r}**). "
        "Hệ số mô tả liên hệ tuyến tính, không phải mức độ quan trọng của biến trong CatBoost. "
        "Với hạng tín dụng, số lớn hơn thể hiện hạng thấp hơn."
    ))
else:
    display(Markdown("**Nhận xét:** lát cắt chưa đủ biến thiên để tính tương quan với rời bỏ."))

## 3. Mô hình rủi ro và phân khúc hành động

CatBoost vẫn dùng đúng 10 predictor, đánh giá ngoài mẫu theo ba fold. Phần này tái sử dụng điểm của mô hình đã chạy; các trường bổ sung trong EDA không phải đầu vào huấn luyện.

### C19a — Điểm rủi ro có phân tầng được khách hàng?

Mỗi nhóm gồm khoảng 10% khách, xếp từ rủi ro thấp đến cao. Cột đỏ có tỷ lệ rời bỏ thực tế vượt đường trung bình của tập khách đang phân tích; cột xanh xám có tỷ lệ bằng hoặc thấp hơn mức này.

In [ ]:
show("C19a_risk_decile")

In [ ]:
risk_groups = c.rate_by(df, "RiskDecile").sort_values("RiskDecile")
display(c.display_table(risk_groups.assign(rate=risk_groups.rate.map(c.fmt_pct))))
highest = risk_groups.iloc[-1]
above_average = risk_groups.loc[risk_groups.rate > c.baseline(df), "RiskDecile"].astype(int)
group_names = ", ".join(above_average.astype(str))
display(Markdown(
    f"**Nhận xét:** nhóm rủi ro {int(highest.RiskDecile)} có tỷ lệ rời bỏ "
    f"**{c.fmt_pct(highest.rate)}**, gấp {c.fmt_ratio(highest.rate / c.baseline(df))} lần trung bình. "
    f"Các nhóm **{group_names}** vượt mức trung bình **{c.fmt_pct(c.baseline(df), 2)}**, nên được tô đỏ. "
    "Màu sắc thể hiện tỷ lệ quan sát thực tế; điểm mô hình dùng để xếp thứ tự các nhóm."
))

### C19b — Với ngân sách hữu hạn, liên hệ bao nhiêu khách là đủ?

In [ ]:
show("C19b_gain_curve")

In [ ]:
ordered = df.sort_values("RiskScore", ascending=False)
total_churners = int(ordered.ChurnFlag.sum())
n10 = max(1, (len(ordered) + 9) // 10)
n20 = max(1, (len(ordered) + 4) // 5)
gain10 = ordered.head(n10).ChurnFlag.sum() / total_churners
gain20 = ordered.head(n20).ChurnFlag.sum() / total_churners
lift10 = ordered.head(n10).ChurnFlag.mean() / c.baseline(df)
display(Markdown(
    f"**Nhận xét:** liên hệ 10% khách có rủi ro cao nhất tiếp cận được **{c.fmt_pct(gain10, 0)}** "
    f"số khách thực tế rời bỏ, hiệu quả gấp **{c.fmt_ratio(lift10)} lần** chọn ngẫu nhiên. "
    f"Liên hệ 20% khách có rủi ro cao nhất tiếp cận được **{c.fmt_pct(gain20, 0)}** số khách rời bỏ."
))

**Quy tắc phân khúc hành động** (xét theo thứ tự, chỉ dùng biến thuộc mô hình):

| # | Phân khúc | Điều kiện |
|---|---|---|
| 1 | Hết hợp đồng | Tháng sử dụng 10–13 |
| 2 | Máy cũ | Máy >730 ngày, hoặc ≥365 ngày và không hỗ trợ web |
| 3 | Rủi ro cao (mô hình) | Còn lại, nhóm rủi ro 8–10 của mô hình 10 biến |
| 4 | Ổn định | Còn lại |

Đã bỏ quy tắc gọi bộ phận giữ chân vì trường đó không thuộc bộ 10; các quy tắc còn lại giữ nguyên điều kiện và thứ tự tương đối.

In [ ]:
seg = c.segment_summary(df)
c.display_table(seg.assign(rate=seg.rate.map(c.fmt_pct), revenue_at_risk=seg.revenue_at_risk.map(c.fmt_money),
                           avg_revenue=seg.avg_revenue.round(1)).drop(columns="revenue"))

### C20 — Phân khúc nào cần ưu tiên, theo rủi ro và giá trị?

In [ ]:
show("C20_phan_khuc")

In [ ]:
intervention = seg[seg.ActionSegment != "Ổn định"]
stable = seg[seg.ActionSegment == "Ổn định"]
largest = intervention.loc[intervention.revenue_at_risk.idxmax()]
display(Markdown(
    f"**Nhận xét:** {len(intervention)} nhóm cần can thiệp gồm **{c.fmt_n(intervention.n.sum())} khách** "
    f"({c.fmt_pct(intervention.n.sum() / len(df), 0)} mẫu). Nhóm ổn định có tỷ lệ rời bỏ "
    f"**{c.fmt_pct(stable.rate.iloc[0])}**. Nhóm **{largest.ActionSegment}** có doanh thu tháng bị đe dọa "
    f"lớn nhất: **{c.fmt_money(largest.revenue_at_risk)}**."
))

### C20b — Nên ưu tiên nhóm nào theo quy mô doanh thu tháng bị đe dọa?

Thanh ngang xếp giảm dần theo **doanh thu tháng bị đe dọa** của các nhóm cần can thiệp. Nhãn cạnh mỗi nhóm ghi số khách và tỷ lệ rời bỏ; nhóm ổn định không nằm trong bảng xếp hạng này.

In [ ]:
# Nạp hai hàm mới nếu kernel đã import charts trước khi file được cập nhật.
import importlib
importlib.reload(c)
show("C20b_uu_tien_doanh_thu")

revenue_priority = c.segment_summary(df)
revenue_priority = revenue_priority[revenue_priority.ActionSegment != "Ổn định"].sort_values("revenue_at_risk", ascending=False)
if not revenue_priority.empty:
    revenue_leader = revenue_priority.iloc[0]
    display(Markdown(
        f"**Nhận xét:** trong các nhóm cần can thiệp, **{revenue_leader.ActionSegment}** đứng đầu theo quy mô doanh thu tháng bị đe dọa: "
        f"**{c.fmt_money(revenue_leader.revenue_at_risk)}**, gồm **{c.fmt_n(revenue_leader.n)} khách**, tỷ lệ rời bỏ **{c.fmt_pct(revenue_leader.rate)}**. "
        "Chỉ số = tổng doanh thu tháng của nhóm × tỷ lệ rời bỏ; đây là quy mô tham khảo, chưa phải doanh thu chắc chắn giữ được."
    ))

### C20c — Nhóm nào có tỷ lệ rời bỏ cao nhất?

Biểu đồ chấm có thân nối (**lollipop**) xếp giảm dần theo **tỷ lệ rời bỏ thực tế**. Chấm càng sang phải, tỷ lệ càng cao; đường đứt là trung bình của toàn bộ mẫu đang xét. Nhóm ổn định được giữ lại làm đối chiếu.

In [ ]:
show("C20c_uu_tien_ty_le")

rate_priority = c.segment_summary(df).sort_values("rate", ascending=False)
if not rate_priority.empty:
    rate_leader = rate_priority.iloc[0]
    display(Markdown(
        f"**Nhận xét:** nhóm **{rate_leader.ActionSegment}** có tỷ lệ rời bỏ cao nhất: **{c.fmt_pct(rate_leader.rate)}**, "
        f"so với trung bình **{c.fmt_pct(c.baseline(df))}**, trên **{c.fmt_n(rate_leader.n)} khách**. "
        "Cách xếp này cho thấy nhóm rời bỏ nhiều nhất theo tỷ lệ; thứ tự có thể khác xếp hạng theo doanh thu ở C20b."
    ))

In [ ]:
policies = {
    "Hết hợp đồng": ("Chăm sóc từ tháng 10; ưu đãi gia hạn 12 tháng trước tháng 11", "Tỷ lệ gia hạn; tỷ lệ rời bỏ tháng 11–13"),
    "Máy cũ": ("Đổi máy trả góp hoặc trợ giá gắn cam kết 12 tháng; ưu tiên máy không hỗ trợ web", "Tỷ lệ nhận đổi máy; tỷ lệ rời bỏ sau 90 ngày; chi phí giữ một khách"),
    "Rủi ro cao (mô hình)": ("Khảo sát hài lòng; tư vấn gói phù hợp; ưu đãi cá nhân hóa", "Tỷ lệ rời bỏ sau 90 ngày; doanh thu giữ được"),
    "Ổn định": ("Duy trì chất lượng và chương trình khách hàng thân thiết", "Theo dõi định kỳ"),
}
rows = ["| Phân khúc | Số khách | Tỷ lệ rời bỏ | Chính sách đề xuất | Chỉ số đo hiệu quả |",
        "|---|---:|---:|---|---|"]
for row in seg.itertuples():
    action, metric = policies[str(row.ActionSegment)]
    rows.append(f"| {row.ActionSegment} | {c.fmt_n(row.n)} | {c.fmt_pct(row.rate)} | {action} | {metric} |")
display(Markdown("## 10. Đề xuất chính sách giữ chân\n\n" + "\n".join(rows) +
                 "\n\n"
                 "Cần thử nghiệm đối chứng ngẫu nhiên trước khi triển khai rộng."))

In [ ]:
display(Markdown(
    "## 11. Tổng kết từ kết quả lần chạy này\n\n"
    f"- Tập nguồn **{c.fmt_n(len(raw))} khách**, loại **{c.fmt_n(removed_count)} dòng** có số âm bất thường, còn **{c.fmt_n(len(df))} khách**.\n"
    f"- Khách rời bỏ **{c.fmt_n(k['churners'])}**, tỷ lệ **{c.fmt_pct(k['rate'], 2)}**; doanh thu tháng của nhóm này **{c.fmt_money(k['lost_revenue'])}**.\n"
    f"- Mô hình dùng **{len(model_info.get('features', []))} biến**, ROC-AUC toàn mẫu ngoài mẫu **{model_info['roc_auc']:.4f}**, AP **{model_info['pr_auc']:.4f}**.\n"
    f"- Liên hệ 10% khách xếp rủi ro cao nhất trong tập EDA tiếp cận **{c.fmt_pct(gain10)}** khách rời bỏ, lift **{c.fmt_ratio(lift10)}×**.\n"
    f"- Các nhóm cần can thiệp gồm **{c.fmt_n(intervention.n.sum())} khách**; nhóm có doanh thu tháng bị đe dọa lớn nhất là **{largest.ActionSegment}** (**{c.fmt_money(largest.revenue_at_risk)}**).\n\n"
    "Các quan hệ là quan sát trên dữ liệu tại một thời điểm; chính sách cần thử nghiệm đối chứng trước khi triển khai rộng."
))

## 12. Xuất kết quả cho bảng điều khiển

Lưu các biểu đồ tương tác đã vẽ và dữ liệu của lần phân tích này vào `work/build/dashboard/`. Bảng điều khiển đọc trực tiếp biểu đồ đã xuất khi hiển thị toàn bộ khách hàng; các bộ lọc dùng bản dữ liệu được lưu cùng lần chạy. Sau khi chạy lại sổ phân tích, khởi động lại bảng điều khiển để nạp kết quả mới.

In [ ]:
RESULTS_DIR = ROOT / "work" / "build" / "dashboard"
RESULT_FIGURES = RESULTS_DIR / "figures"
RESULT_FIGURES.mkdir(parents=True, exist_ok=True)
for stale_path in RESULT_FIGURES.glob("*.json"):
    if stale_path.stem not in c.FIGURES and stale_path.resolve().parent == RESULT_FIGURES.resolve():
        stale_path.unlink()

for key, figure in dashboard_figures.items():
    figure.write_json(RESULT_FIGURES / f"{key}.json")
df.to_parquet(RESULTS_DIR / "customers.parquet", index=False)
print(f"Đã xuất {len(dashboard_figures)} biểu đồ và {c.fmt_n(len(df))} khách hàng cho bảng điều khiển.")

# Đồng bộ tài liệu với đúng dữ liệu và điểm mô hình của lần chạy này.
report_lines = [
    "# Insight và chính sách giữ chân khách hàng",
    "",
    f"Nguồn: {c.fmt_n(len(raw))} khách; sau khi loại {c.fmt_n(removed_count)} dòng có số âm bất thường, tập EDA còn **{c.fmt_n(len(df))} khách**, **{c.fmt_n(k['churners'])} khách rời bỏ ({c.fmt_pct(k['rate'], 2)})**.",
    "",
    f"Mô hình **{model_info['model_version']}** chỉ dùng **{len(model_info['features'])} biến đầu vào**. Mô hình và EDA dùng cùng bộ 10 biến; các band và điểm rủi ro được suy ra từ chúng.",
    "",
    "## Các phát hiện từ dữ liệu hiện hành",
    "",
    "| Biểu đồ | Nhận xét tính từ dữ liệu |",
    "|---|---|",
]
for key in ['C01_ty_le_churn', 'C05_vong_doi', 'C06_tuoi_thiet_bi', 'C07_heatmap_tenure_thiet_bi', 'C04b_phan_bo_doanh_thu', 'C04d_churn_phi_dinh_ky', 'C12_muc_su_dung', 'C10_bien_dong_phut_goi', 'C11_bien_dong_doanh_thu', 'C13_goi_gio_cao_diem', 'C14_treemap_thoi_gian', 'C16_tin_dung', 'C18_tuong_quan', 'C19b_gain_curve']:
    title = dashboard_figures[key].layout.title.text.split("<br>")[0].replace("<b>", "").replace("</b>", "")
    report_lines.append(f"| [{key.split('_')[0]}](reports/figures/{key}.png) | {title} |")
report_lines += [
    "",
    "## Kết quả mô hình 10 biến",
    "",
    f"Đánh giá ngoài mẫu trên {c.fmt_n(len(raw))} khách nguồn: ROC-AUC **{model_info['roc_auc']:.4f}**, AP **{model_info['pr_auc']:.4f}**, lift top 10% **{c.fmt_ratio(model_info['capacity_10']['lift'])}×**, recall top 20% **{c.fmt_pct(model_info['capacity_20']['recall'])}**.",
    "",
    f"Trong tập EDA sau xử lý: lift top 10% **{c.fmt_ratio(lift10)}×**, gain top 20% **{c.fmt_pct(gain20)}**. Hai tập có mẫu số khác nhau do bước loại số âm bất thường.",
    "",
    "## Phân khúc và chính sách",
    "",
    "| Phân khúc | Số khách | Tỷ lệ rời bỏ | Doanh thu tháng bị đe dọa | Chính sách đề xuất | KPI |",
    "|---|---:|---:|---:|---|---|",
]
for row in seg.itertuples():
    action, metric = policies[str(row.ActionSegment)]
    report_lines.append(f"| {row.ActionSegment} | {c.fmt_n(row.n)} | {c.fmt_pct(row.rate)} | {c.fmt_money(row.revenue_at_risk)} | {action} | {metric} |")
report_lines += [
    "",
    f"Các nhóm cần can thiệp gồm **{c.fmt_n(intervention.n.sum())} khách**; doanh thu tháng bị đe dọa **{c.fmt_money(intervention.revenue_at_risk.sum())}**.",
    "",
    "Doanh thu tháng bị đe dọa = tổng doanh thu tháng của nhóm × tỷ lệ rời bỏ; chưa phải doanh thu chắc chắn giữ được. Dữ liệu tại một thời điểm không chứng minh nhân quả; chính sách cần thử nghiệm đối chứng.",
]
(ROOT / "INSIGHTS_AND_POLICY.md").write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("Đã đồng bộ INSIGHTS_AND_POLICY.md với kết quả 10 biến.")
